<a href="https://colab.research.google.com/github/Sonpons/COPOD-reproduction-assignment-2/blob/main/Assignment%204%20ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:

# install libraries
%pip install aif360 scikit-learn pandas numpy

# import
import numpy as np
import pandas as pd
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import Reweighing
from aif360.metrics import BinaryLabelDatasetMetric
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# 1. load all the datasets (Adult, German and COMPAS)

# Adult
#positive outcome:income above $50K
adult = pd.read_csv("adult.csv")
adult["label"] = (adult["income"] == ">50K").astype(int)
adult["protected"] = (adult["sex"] == "Male").astype(int)
adult = adult.drop(columns=["income", "sex"])

# German
#positive outcome = good credit
german = pd.read_csv("german.data", sep=r"\s+", header=None)
german.columns = [f"column_{i}" for i in range(21)]
german["label"] = (german["column_20"] == 1).astype(int)
german["protected"] = (german["column_12"] > 25).astype(int)
german = german.drop(columns=["column_20"])

# COMPAS
# positive outcome = no reoffending
compas = pd.read_csv("propublica_data_for_fairml.csv")
compas["label"] = 1 - compas["Two_yr_Recidivism"]
compas["protected"] = 1 - compas["African_American"]
compas = compas.drop(columns=[
    "Two_yr_Recidivism", "African_American", "score_factor"
])

datasets = {
    "Adult": adult,
    "German": german,
    "COMPAS": compas
}


# 2. Fairness

def fairness(data):
    metric = BinaryLabelDatasetMetric(
        data,
        privileged_groups=[{"protected": 1}],
        unprivileged_groups=[{"protected": 0}]
    )
    return {
        "DP_Diff": metric.statistical_parity_difference(),
        "DIR": metric.disparate_impact(),
        "Consistency": metric.consistency()
    }


# 3. Running it on the 3 different datasets

for name, df in datasets.items():

    print("\nDATASET:", name)

    # convert categorical to numbers
    df = pd.get_dummies(df, dtype=int)

    # create AIF360 dataset
    data = BinaryLabelDataset(
        df=df.astype(float),
        label_names=["label"],
        protected_attribute_names=["protected"]
    )

    # fairness before mitigation
    original = fairness(data)

    # stratified train/test split 70-30
    strata = (
        df["protected"].astype(str) + "_" +
        df["label"].astype(str)
    )

    train_idx, test_idx = train_test_split(
        np.arange(len(df)),
        test_size=0.3,
        random_state=42,
        stratify=strata
    )

    train = data.subset(train_idx.tolist())
    test = data.subset(test_idx.tolist())

    # original fairness on test set
    original_test = fairness(test)

    # prepare training and test data
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train.features)
    X_test = scaler.transform(test.features)

    y_train = train.labels.ravel()
    y_test = test.labels.ravel()

    # train classifier without mitigation
    baseline = LogisticRegression(max_iter=2000)
    baseline.fit(X_train, y_train)
    baseline_pred = baseline.predict(X_test)

    baseline_data = test.copy(deepcopy=True)
    baseline_data.labels = baseline_pred.reshape(-1, 1)

    baseline_results = fairness(baseline_data)
    baseline_results["Accuracy"] = accuracy_score(
        y_test, baseline_pred
    )

    #  Reweighing
    rw = Reweighing(
        unprivileged_groups=[{"protected": 0}],
        privileged_groups=[{"protected": 1}]
    )

    weighted_train = rw.fit_transform(train)

    # train the classifier with Reweighing
    model = LogisticRegression(max_iter=2000)
    model.fit(
        X_train,
        y_train,
        sample_weight=weighted_train.instance_weights
    )

    # prediction on test data
    predictions = model.predict(X_test)

    # calculate the fairness after mitigation
    result_data = test.copy(deepcopy=True)
    result_data.labels = predictions.reshape(-1, 1)

    after = fairness(result_data)
    after["Accuracy"] = accuracy_score(y_test, predictions)

    # comparison of results
    results = pd.DataFrame({
        "Original labels (full)": original,
        "Original labels (test)": original_test,
        "Without Reweighing": baseline_results,
        "After Reweighing": after
    }).T

    print(results.round(3))



DATASET: Adult
                         DP_Diff       DIR           Consistency  Accuracy
Original labels (full) -0.196276  0.358023  [0.7696139553453878]       NaN
Original labels (test) -0.196268  0.358177  [0.7565359811649068]       NaN
Without Reweighing     -0.170348  0.341087  [0.8072883611423818]  0.848705
After Reweighing       -0.073577  0.651217  [0.8195925888013033]  0.842358

DATASET: German
                         DP_Diff       DIR           Consistency  Accuracy
Original labels (full) -0.149448  0.794826  [0.6808000000000012]       NaN
Original labels (test) -0.149448  0.794826  [0.6619999999999995]       NaN
Without Reweighing     -0.150097  0.823799  [0.7786666666666664]      0.77
After Reweighing       -0.036604  0.956611  [0.7879999999999998]  0.776667

DATASET: COMPAS
                         DP_Diff       DIR           Consistency  Accuracy
Original labels (full)   -0.1401  0.772915  [0.6093972780298118]       NaN
Original labels (test) -0.139776  0.773336  [0.622

In [ ]:
from google.colab import drive
drive.mount('/content/drive')